In [35]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from features.category_probability_extractor import CategoryProbabilityExtractor
from features.brand_imputer import BrandImputer
from features.pack_size_probability_extractor import PackSizeProbabilityExtractor
from features.data_preprocessor import DataPreprocessor
from features.cross_validation_ensemble import CrossValidationEnsemble
from sklearn.ensemble import HistGradientBoostingClassifier

In [36]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

category probability

In [37]:
base_lr = LogisticRegression(max_iter=300, random_state=42)
extractor = CategoryProbabilityExtractor(base_model=base_lr, n_splits=5)
extractor.fit(train)

              precision    recall  f1-score   support

       audio       0.90      0.96      0.93      2155
      beauty       1.00      0.97      0.98      1364
       books       0.92      0.99      0.95      1462
   computers       0.97      0.97      0.97      2318
      garden       0.97      0.96      0.97      1050
     kitchen       0.95      0.95      0.95      1864
       music       0.94      0.55      0.69       466
        pets       1.00      0.97      0.99       714
       sport       0.94      0.93      0.94      1580

    accuracy                           0.95     12973
   macro avg       0.95      0.92      0.93     12973
weighted avg       0.95      0.95      0.95     12973



In [38]:
train = extractor.transform(train)
train['declared_category'] = train['declared_category'].fillna('missing')

test = extractor.transform(test)
test['declared_category'] = test['declared_category'].fillna('missing')

brands

In [39]:
bi = BrandImputer(['Noro', 'Velto', 'Aster', 'Mira', 'Tenso'])
train = bi.transform(train)
test = bi.transform(test)

In [40]:
model = LogisticRegression(C=10.0, max_iter=1000)
ppe = PackSizeProbabilityExtractor(model)
ppe.fit(train)
train = ppe.transform(train)
test = ppe.transform(test)

              precision    recall  f1-score   support

         1.0       1.00      1.00      1.00      9329
         2.0       1.00      1.00      1.00      2038
         3.0       1.00      1.00      1.00      1000
         5.0       1.00      1.00      1.00       515

    accuracy                           1.00     12882
   macro avg       1.00      1.00      1.00     12882
weighted avg       1.00      1.00      1.00     12882



In [41]:
CATEGORICAL_FEATURES = ['declared_category', 'predicted_category', 'brand', 'condition']
NUMERICAL_FEATURES = [
    'declared_cat_prob', 'declared_pack_size_prob',
    'cat_prob_delta', 'pack_prob_delta', 'category_mismatch', 'pack_size_mismatch',
    'z_score_price', 'z_score_price_abs', 'z_score_weight', 'z_score_weight_abs',
    'log_unit_price', 'log_unit_weight', 'log_price', 'log_weight',
    'pack_size', 'predicted_pack_size',
    'title_len', 'desc_len', 'has_missing_price', 'has_missing_weight'
]
FEATURES = CATEGORICAL_FEATURES + NUMERICAL_FEATURES

dp = DataPreprocessor()
base_model = HistGradientBoostingClassifier(
    categorical_features=CATEGORICAL_FEATURES,
    max_iter=300,
    learning_rate=0.04,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=42
)
cve = CrossValidationEnsemble(
    base_model=base_model,
    preprocessor=dp,
    features=FEATURES,
    n_splits=5
)

In [42]:
cve.fit(train)

--- Fold 1 ---
ROC-AUC: 0.9058 | PR-AUC: 0.8139
--- Fold 2 ---
ROC-AUC: 0.8978 | PR-AUC: 0.8126
--- Fold 3 ---
ROC-AUC: 0.9035 | PR-AUC: 0.8352
--- Fold 4 ---
ROC-AUC: 0.9100 | PR-AUC: 0.8316
--- Fold 5 ---
ROC-AUC: 0.9083 | PR-AUC: 0.8228

OOF ROC-AUC : 0.9051
OOF PR-AUC  : 0.8220


In [44]:
test_preds = cve.predict_proba(test)

In [46]:
submission = test[['id']].copy()
submission['target'] = test_preds
submission.to_csv('submission.csv', index=False)